In [0]:
import pandas as pd
from pyspark.sql.functions import col, lit, when

#Global

In [0]:
tb_moni = 'ia.tb_diamond_mod_monitoramento'

tb_entrada = 'ia.tb_diamond_mod_pulmao'
tb_saida = 'ia.tb_diamond_mod_pulmao_saida'
tb_envio = 'ia.tb_diamond_mod_pulmao_saida_salesforce'

In [0]:
df_entrada = spark.sql(f"""
    select 
        cast(dataExecucaoModelo as date) as dt_execucao,
        nme_regional_hospital as nm_regional,
        COUNT(an) AS qtd_entrada,
        unidade as nm_unidade
    from {tb_entrada} 
    where nme_regional_hospital is not null 
    and dataExecucaoModelo >= '2026-01-28'
    GROUP BY dataExecucaoModelo, nme_regional_hospital, unidade
    order by dataExecucaoModelo
""")
df_saida = spark.sql(f"""
    select 
        cast(dataExecucaoModelo as date) as dt_execucao,
        nme_regional_hospital as nm_regional,
        COUNT(exm_an) AS qtd_saida,
        unidade as nm_unidade
    from {tb_saida} 
    where nme_regional_hospital is not null
    and dataExecucaoModelo >= '2026-01-28'
    GROUP BY dataExecucaoModelo, nme_regional_hospital, unidade
    order by dataExecucaoModelo
""")
df_envio = spark.sql(f"""
    select 
        cast(dataExecucaoModelo as date) as dt_execucao,
        nme_regional_hospital as nm_regional,
        COUNT(exm_an) AS qtd_envio,
        unidade as nm_unidade
    from {tb_envio} 
    where nme_regional_hospital is not null
    and dataExecucaoModelo >= '2026-01-28'
    and exm_encaminhamento_nlp = "S" 
    GROUP BY dataExecucaoModelo, nme_regional_hospital, unidade
    order by dataExecucaoModelo
""")

In [0]:
df_monitoramento = (
    df_entrada.alias("e")
        .join(df_saida.alias("s"), on=["dt_execucao", "nm_regional", "nm_unidade"], how="left")
        .join(df_envio.alias("r"), on=["dt_execucao", "nm_regional", "nm_unidade"], how="left")
        .select(
            "dt_execucao",
            "e.qtd_entrada",
            "s.qtd_saida",
            "r.qtd_envio",
            "e.nm_regional", 
            "e.nm_unidade",
        )
)

In [0]:
df_monitoramento = (
    df_monitoramento
        .withColumn("qtd_saida", col("qtd_saida").cast("int"))
        .withColumn("qtd_entrada", col("qtd_entrada").cast("int"))
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
        .withColumn("nm_projeto", lit("Pulmao").cast("string"))
)

In [0]:
df_monitoramento.createOrReplaceTempView("vw_monitoramento_wrk_01")

In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_wrk_01 s
ON  t.dt_execucao = s.dt_execucao
AND t.nm_regional = s.nm_regional
AND t.nm_unidade = s.nm_unidade
AND t.nm_projeto  = s.nm_projeto

WHEN NOT MATCHED THEN
  INSERT (
    dt_execucao,
    qtd_entrada,
    qtd_saida,
    qtd_envio,
    nm_regional,
    nm_unidade,
    nm_projeto
  )
  VALUES (
    s.dt_execucao,
    s.qtd_entrada,
    s.qtd_saida,
    s.qtd_envio,
    s.nm_regional,
    s.nm_unidade,
    s.nm_projeto
  )


In [0]:
dbutils.notebook().exit("Fim da execução!")

In [0]:
%sql
select 
dt_execucao,
qtd_entrada,
qtd_saida,
qtd_envio,
nm_regional,
nm_unidade
from ia.tb_diamond_mod_monitoramento 
where nm_projeto = "Pulmao" 
and nm_unidade is not null
and qtd_envio > 0
--and nm_regional = "RJ" 
and dt_execucao >= "2026-02-01"
order by dt_execucao desc